# Spider Text-to-SQL — Schema + Question + Query Join

Purpose: produce a single training-ready file where every row is
`(schema_string, question, query)`, ready to slot into the prompt
template from Stage 5 of the pipeline.

Two inputs, joined on `db_id`:
- **HF `xlangai/spider`** → `question`, `query`, `db_id`
- **Official Spider1 `tables.json`** → schema for every `db_id`

Edit the path in the cell below to wherever you unzipped the
official Spider1 data — everything else runs as-is.

In [1]:
from pathlib import Path
import json
import pandas as pd
from datasets import load_dataset

# --- EDIT THIS to your local SPIDER_DATA folder ---
SPIDER_OFFICIAL_DIR = Path("SPIDER_DATA")
TABLES_JSON = SPIDER_OFFICIAL_DIR / "data" / "tables.json"

assert TABLES_JSON.exists(), f"Can't find {TABLES_JSON} — fix SPIDER_OFFICIAL_DIR above"


AssertionError: Can't find SPIDER_DATA\data\tables.json — fix SPIDER_OFFICIAL_DIR above

## 1. Load HF spider (question / query / db_id)

Same as your EDA notebook.

In [ ]:
ds = load_dataset("xlangai/spider")
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
train_df["split"] = "train"
val_df["split"] = "validation"
df = pd.concat([train_df, val_df], ignore_index=True)

print(f"train: {len(train_df)}  val: {len(val_df)}  total: {len(df)}")


## 2. Remove duplicates

Generalized version of what you already did in `SQL_data.ipynb` —
using `subset=` instead of your hardcoded index list, so this keeps
working if the underlying HF dataset version ever shifts row order.

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=["db_id", "question", "query"], keep="first").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows (had {before}, now {len(df)})")


## 3. Load the official schema catalog

In [ ]:
with open(TABLES_JSON, "r", encoding="utf-8") as f:
    all_schemas = json.load(f)

# index by db_id for O(1) lookup
schema_by_db = {s["db_id"]: s for s in all_schemas}
print(f"Loaded schemas for {len(schema_by_db)} databases")

missing = set(df["db_id"].unique()) - set(schema_by_db.keys())
assert not missing, f"HF has db_id(s) with no schema entry: {missing}"
print("Every db_id in the HF data has a matching schema. Good.")


## 4. Manual sanity check — inspect ONE db_id by hand first

Do not skip this. Pick one small database and look at the raw
`tables.json` structure before trusting any function to transform
it. This is the "derive it on paper" step.

In [ ]:
sample_db_id = df["db_id"].value_counts().index[-1]  # smallest db, easiest to read by hand
print("Inspecting:", sample_db_id)
print(json.dumps(schema_by_db[sample_db_id], indent=2))


Look at that output and answer these yourself before moving on:
- What does `column_names_original[0]` look like, and why?
- What do the integers in `primary_keys` refer to?
- What do the two integers in each `foreign_keys` pair mean?

If you can't answer those from the printed JSON, re-read Stage 2 of
the pipeline doc before running the next cell.

## 5. Schema linearization function

Converts one `tables.json` entry into a prompt-ready string.
Two styles:
- `"flat"` → `table(col1, col2, ...)` — compact
- `"ddl"`  → `CREATE TABLE ...` — recommended, matches Qwen2.5-Coder's
  code-pretraining distribution much more closely than the flat form.

This was unit-tested separately against a hand-built fake schema
before being dropped in here — the FK/PK resolution logic is the
part most likely to silently misbehave if you edit it, so re-test
on a toy schema if you change it.

In [ ]:
def linearize_schema(schema, style="ddl"):
    table_names = schema["table_names_original"]
    col_names = schema["column_names_original"]   # [[table_idx, col_name], ...], -1 = "*"
    col_types = schema["column_types"]
    pk_idxs = set(schema["primary_keys"])
    fks = schema["foreign_keys"]                  # [[child_col_idx, parent_col_idx], ...]

    tables = {i: [] for i in range(len(table_names))}
    for global_idx, (t_idx, col_name) in enumerate(col_names):
        if t_idx == -1:
            continue  # skip the "*" placeholder column
        tables[t_idx].append((global_idx, col_name, col_types[global_idx]))

    idx_to_col = {}
    for t_idx, cols in tables.items():
        for global_idx, col_name, _ in cols:
            idx_to_col[global_idx] = (table_names[t_idx], col_name)

    if style == "flat":
        lines = []
        for t_idx, cols in tables.items():
            col_str = ", ".join(c[1] for c in cols)
            lines.append(f"{table_names[t_idx]}({col_str})")
        return "\n".join(lines)

    type_map = {"text": "TEXT", "number": "INTEGER", "time": "DATETIME",
                "boolean": "BOOLEAN", "others": "TEXT"}
    ddl_statements = []
    for t_idx, cols in tables.items():
        col_lines = []
        for global_idx, col_name, col_type in cols:
            line = f"  {col_name} {type_map.get(col_type, 'TEXT')}"
            if global_idx in pk_idxs:
                line += " PRIMARY KEY"
            col_lines.append(line)
        for child_idx, parent_idx in fks:
            if child_idx in [c[0] for c in cols]:
                child_col = idx_to_col[child_idx][1]
                parent_table, parent_col = idx_to_col[parent_idx]
                for i, l in enumerate(col_lines):
                    if l.strip().startswith(child_col + " "):
                        col_lines[i] += f" REFERENCES {parent_table}({parent_col})"
        ddl = f"CREATE TABLE {table_names[t_idx]} (\n" + ",\n".join(col_lines) + "\n);"
        ddl_statements.append(ddl)
    return "\n".join(ddl_statements)


# sanity check on the same db_id you just inspected by hand
print(linearize_schema(schema_by_db[sample_db_id], style="ddl"))


Compare this printed output line-by-line against the raw JSON from
Stage 4. If a foreign key looks attached to the wrong column, or a
primary key is missing, stop here and debug before joining anything.

## 6. Build a schema-string cache for every db_id

Do this once, not per-row — `tables.json` is small, no need to
re-parse it 8000 times.

In [ ]:
schema_cache = {
    db_id: linearize_schema(schema, style="ddl")
    for db_id, schema in schema_by_db.items()
}
print(f"Built {len(schema_cache)} linearized schemas")


## 7. Join: attach schema string to every row

In [ ]:
df["schema_str"] = df["db_id"].map(schema_cache)
assert df["schema_str"].isnull().sum() == 0, "Some rows didn't get a schema — check db_id matching"
df[["db_id", "question", "query", "schema_str"]].head(2)


## 8. Assemble the final training prompt

Kept `prompt` (input) and `query` (target) as separate columns —
not one merged string — so your training script can mask the loss
to the completion only, same correctness issue you already caught
in the financial-Chat DPO pipeline.

In [ ]:
PROMPT_TEMPLATE = """### Schema:
{schema}

### Question:
{question}

### SQL:"""

df["prompt"] = df.apply(
    lambda r: PROMPT_TEMPLATE.format(schema=r["schema_str"], question=r["question"]),
    axis=1
)

print(df.loc[0, "prompt"])
print("\n--- target ---")
print(df.loc[0, "query"])


## 9. Re-check sequence length now that schema is included

Your original EDA measured question/query length alone. That
understated the real input size — redo it here on `prompt`.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-14B-Instruct")

df["prompt_tokens"] = df["prompt"].apply(lambda s: len(tokenizer.encode(s)))
df["query_tokens"] = df["query"].apply(lambda s: len(tokenizer.encode(s)))
df["total_tokens"] = df["prompt_tokens"] + df["query_tokens"]

print(df["total_tokens"].describe(percentiles=[.5, .9, .95, .99]))
print(f"\nRows exceeding 4096 tokens: {(df['total_tokens'] > 4096).sum()}")


If that last number isn't 0, your Stage 0 context-length assumption
needs revisiting — either truncate/filter those rows, switch to the
flat schema style to save tokens, or bump the context window.

If you don't have internet access to pull the tokenizer, a rough
word-count proxy (`len(s.split())`) will underestimate real token
count for code/SQL — don't rely on it for the final decision, only
as a quick offline sanity check.

## 10. Save the processed, joined dataset

In [ ]:
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

cols_to_save = ["db_id", "question", "schema_str", "prompt", "query", "split"]

train_out = df[df["split"] == "train"][cols_to_save]
val_out = df[df["split"] == "validation"][cols_to_save]

train_out.to_json(out_dir / "train_assembled.jsonl", orient="records", lines=True, force_ascii=False)
val_out.to_json(out_dir / "val_assembled.jsonl", orient="records", lines=True, force_ascii=False)

print(f"Saved {len(train_out)} train rows and {len(val_out)} val rows to {out_dir}/")
